<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Module_11_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consider:

x = torch.tensor([2.0], requires_grad=True)

y = x**2 + 3*x + 1

y.backward()

What is the value inside x.grad?

Explain how PyTorch computed it step-by-step.


w ──┐

       ├─► z (w*x + b) ──► y (z²) ──► L

x ──┤

b ──┘


```text
x.grad: 7.0


```

The value inside `x.grad` is **`tensor([7.])`** (or the scalar float value **7.0**).

**Mathematical Derivation**

PyTorch computes the derivative of the scalar function $y$ with respect to $x$:

$$y = x^2 + 3x + 1$$

Taking the derivative using standard calculus rules:

$$\frac{dy}{dx} = \frac{d}{dx}(x^2) + \frac{d}{dx}(3x) + \frac{d}{dx}(1) = 2x + 3$$

Evaluating the derivative at $x = 2.0$:

$$\left. \frac{dy}{dx} \right\vert{}_{x=2.0} = 2(2.0) + 3 = 4.0 + 3 = 7.0$$

**PyTorch Execution Mechanics**

PyTorch computes this result using reverse-mode automatic differentiation (**autograd**):

1. **Forward Pass (Graph Construction):**
* Setting `requires_grad=True` signals PyTorch to track all operations involving `x` and build a dynamic Directed Acyclic Graph (DAG).
* It breaks down the expression into elementary operations:
* $u = x^2 = (2.0)^2 = 4.0$
* $v = 3x = 3 \times 2.0 = 6.0$
* $y = u + v + 1 = 4.0 + 6.0 + 1 = 11.0$


* Each tensor created during this pass retains a reference to its gradient function (`grad_fn`).


2. **Backward Pass (Chain Rule & Gradient Accumulation):**
* Calling `y.backward()` initializes a seed gradient of $\frac{\partial y}{\partial y} = 1.0$ at the root node.
* Autograd traverses backward through the DAG using the chain rule:
* Gradient from the $x^2$ branch: $\frac{\partial u}{\partial x} = 2x \implies 2(2.0) = 4.0$
* Gradient from the $3x$ branch: $\frac{\partial v}{\partial x} = 3$
* Constant term $1$ has a derivative of $0$.


* Because `x` appears in multiple terms ($x^2$ and $3x$), the incoming gradient paths are summed at the leaf node:

$$\text{x.grad} = 4.0 + 3.0 = 7.0$$




3. **Storage:**
* The calculated gradient is populated into the `.grad` attribute of the leaf tensor `x`.


# What happens in this case?

 x = torch.tensor([2.0])
 y = x**2

 y.backward()

 Why does it fail?


It crashes and throws a **`RuntimeError`**:

```text
RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

```

### Why It Fails

1. **`x` doesn't track gradients:** By default, PyTorch creates tensors with `requires_grad=False`.
2. **No calculation history is saved:** Because `x` isn't tracking history, PyTorch doesn't bother building a computational graph when computing `y = x**2`.
3. **`y.grad_fn` is `None`:** When you call `y.backward()`, PyTorch looks for the math history (the computational graph) to figure out how to do calculus backward. Because there is no record of how `y` was created, it hits a dead end and crashes.

### How to Fix It

Tell PyTorch to track operations on `x` from the start:

```python
x = torch.tensor([2.0], requires_grad=True)  # <-- Added requires_grad=True
y = x**2

y.backward()  # Works now!
print(x.grad) # tensor([4.])

```